In [2]:
import pandas as pd
import ast
from collections import Counter
import time


def safe_parse_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

csv_path = "../data/dataset.csv"
df = pd.read_csv(csv_path, low_memory=False, nrows=10000)
df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [3]:
# Extract all ingredients across the full dataset (chunked, memory-safe)
ingredient_counter = Counter()
chunk_size = 100_000

for chunk in pd.read_csv(csv_path, usecols=["ingredients"], chunksize=chunk_size, low_memory=False):
    parsed_lists = chunk["ingredients"].map(safe_parse_list)
    for ing_list in parsed_lists:
        cleaned = [str(item).strip() for item in ing_list if str(item).strip()]
        ingredient_counter.update(cleaned)

all_ingredients = sorted(ingredient_counter.keys(), key=str.lower)
print(f"Unique ingredients found: {len(all_ingredients):,}")
print(f"Total ingredient mentions: {sum(ingredient_counter.values()):,}")

Unique ingredients found: 4,678,055
Total ingredient mentions: 19,471,236


In [4]:
# Quick inspection of extracted ingredient universe
ingredients_df = (
    pd.Series(ingredient_counter, name="count")
    .sort_values(ascending=False)
    .rename_axis("ingredient")
    .reset_index()
    .head(30)
)

display(ingredients_df)
print("First 25 ingredients (alphabetical):")
print(all_ingredients[:25])

,ingredient,count
0,2 eggs,95871
1,1 tsp. vanilla,85291
2,1/2 tsp. salt,80654
3,1 tsp. salt,80377
4,1 egg,77892
5,1/2 teaspoon salt,70459
6,1 c. sugar,65051
7,1 teaspoon salt,63555
8,1/4 teaspoon salt,47685
9,2 c. sugar,44174


First 25 ingredients (alphabetical):
['! 1/3 C fresh lemon juice', '! 2 tablespoons cold butter', '! 2 Tbs dry white wine', '! 2 tbs minced garlic', '! 2 tbs minced yellow onions', '! 2 Tbs. soy sauce', '! bunches parsley', '! C Creamy Peanut Butter', '! C. green apple (Granny Smith, peeled, cored and chopped', '! clove garlic smashed', '! clove garlic, crushed', '! Cup Brown Sugar', '! cup butter', '! cup grated Paremesan Cheese', '! cup soy sauce', '! cup sugar (I use less)', '! ht fat free sour cream', '! kosher salt or sea salt', '! large rock-firm mango, peeled, seeded, and finely chopped', '! lb lean pork tenderloin, fat trimmed.', '! medium red onion, chopped', "! pkg. Knorr's Soup mix", '! pounds Ground Round', '! Tablespoon Each', '! tablespoon grated fresh ginger']


In [5]:
from ingredient_parser import parse_ingredient

def normalize_ingredients(raw_ingredients):
    parsed_tuples = []
    for raw in raw_ingredients:
        parsed = parse_ingredient(raw)
        item = parsed.name[0].text if len(parsed.name) > 0 else None

        quantity = None
        unit = None

        if parsed.amount:
            amount_obj = parsed.amount[0]
            quantity_value = getattr(amount_obj, "quantity", None)
            if quantity_value is not None:
                try:
                    quantity = str(float(quantity_value))
                except ValueError:
                    quantity = str(quantity_value)

            unit_value = getattr(amount_obj, "unit", None)
            if unit_value is not None:
                unit = str(unit_value)

        parsed_tuples.append((item, quantity, unit))
    return parsed_tuples

demo_recipe = pd.read_csv(csv_path, nrows=10, low_memory=False).iloc[0]
raw_ingredients = ast.literal_eval(demo_recipe["ingredients"])
normalize_ingredients(raw_ingredients)

ModuleNotFoundError: No module named 'ingredient_parser'

In [ ]:
# Apply normalize_ingredients row by row for easier debugging
print("Normalizing ingredients row by row...")

normalized_ingredients = []
t0 = time.time()
for idx, raw_value in enumerate(df["ingredients"]):
    try:
        raw_ingredients = ast.literal_eval(raw_value) if isinstance(raw_value, str) else raw_value
        normalized_row = normalize_ingredients(raw_ingredients)
        normalized_ingredients.append(normalized_row)

        if idx < 5:
            print(f"Row {idx} OK -> {normalized_row[:3]}")
    except Exception as e:
        normalized_ingredients.append([])
        print(f"Row {idx} failed: {e}")

    if idx > 0 and idx % 1000 == 0:
        print(f"Processed {idx} rows in {time.time() - t0:.2f} seconds...")

df["normalized_ingredients"] = normalized_ingredients

print("Done.")
print(df[["title", "normalized_ingredients"]].head(3).to_string(index=False))
print(f"Total time taken: {time.time() - t0:.2f} seconds")

Normalizing ingredients row by row...
Row 0 OK -> [('brown sugar', '1.0', 'cup'), ('evaporated milk', '0.5', 'cup'), ('vanilla', '0.5', 'teaspoon')]
Row 1 OK -> [('beef', '1.0', 'small jar'), ('chicken breasts', '4.0', ''), ('cream of mushroom soup', '1.0', 'can')]
Row 2 OK -> [('pkg. frozen corn', '2.0', ''), ('pkg. cream cheese', '1.0', ''), ('butter', '0.3333333333333333', 'cup')]
Row 3 OK -> [('whole chicken', '1.0', ''), ('chicken gravy', '2.0', 'cans'), ('cream of mushroom soup', '1.0', 'can')]
Row 4 OK -> [('peanut butter', '1.0', 'cup'), ('graham cracker crumbs', '0.75', 'cup'), ('butter', '1.0', 'cup')]
Processed 1000 rows in 4.25 seconds...
Processed 2000 rows in 8.45 seconds...
Processed 3000 rows in 12.60 seconds...
Processed 4000 rows in 16.73 seconds...
Processed 5000 rows in 20.70 seconds...
Processed 6000 rows in 24.76 seconds...
Processed 7000 rows in 28.93 seconds...
Processed 8000 rows in 33.18 seconds...
Processed 9000 rows in 37.56 seconds...
Done.
                

### Build `ner_tokens` for filtering
This expands ingredient names into lowercase tokens so filtering can match terms like “cheese” inside “italian cheese”.

Example:
- Input: `[("italian cheese", None, None), ("tomato sauce", None, None)]`
- Output: `["cheese", "italian", "sauce", "tomato"]`

In [ ]:
import re

def _tokenize_name(name: str) -> list[str]:
    tokens = re.split(r"[^a-z]+", name.lower())
    return [t for t in tokens if t]

def build_ner_tokens(normalized_rows: list[list[tuple[str | None, str | None, str | None]]]) -> list[list[str]]:
    output: list[list[str]] = []
    for row in normalized_rows:
        token_set: set[str] = set()
        for item, _, _ in row:
            if not item:
                continue
            token_set.update(_tokenize_name(str(item)))
        output.append(sorted(token_set))
    return output

df["ner_tokens"] = build_ner_tokens(df["normalized_ingredients"].tolist())

print(df[["title", "normalized_ingredients", "ner_tokens"]].head(3).to_string(index=False))

In [ ]:
df.to_csv("../data/dataset_normalized_10000.csv", index=False)

In [ ]:
df

,Unnamed: 0,title,ingredients,directions,link,source,NER,normalized_ingredients
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","[(brown sugar, 1.0, cup), (evaporated milk, 0...."
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","[(beef, 1.0, small jar), (chicken breasts, 4.0..."
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...","[(pkg. frozen corn, 2.0, ), (pkg. cream cheese..."
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...","[(whole chicken, 1.0, ), (chicken gravy, 2.0, ..."
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu...","[(peanut butter, 1.0, cup), (graham cracker cr..."
...,...,...,...,...,...,...,...,...
9995,9995,Pink Fruit Salad,"[""1 can cherry pie filling"", ""1 can sweetened ...","[""Combine all ingredients and chill.""]",www.cookbooks.com/Recipe-Details.aspx?id=183170,Gathered,"[""cherry pie filling"", ""condensed milk"", ""pine...","[(cherry pie filling, 1.0, can), (sweetened co..."
9996,9996,Peppered Steak,"[""1 lb. round steak"", ""1 bell pepper"", ""1 onio...","[""Cut steaks into strips; brown in cooking oil...",www.cookbooks.com/Recipe-Details.aspx?id=462037,Gathered,"[""bell pepper"", ""onion"", ""tomatoes"", ""salt"", ""...","[(round steak, 1.0, pound), (bell pepper, 1.0,..."
9997,9997,Chicken Casserole,"[""3 lb. fryer"", ""1 large onion"", ""1 large gree...","[""Stew and bone fryer."", ""Saute in small amoun...",www.cookbooks.com/Recipe-Details.aspx?id=292083,Gathered,"[""fryer"", ""onion"", ""green pepper"", ""celery"", ""...","[(fryer, 3.0, pound), (onion, 1.0, ), (green p..."
9998,9998,Sweet Potatoes Casserole,"[""1 large can yams"", ""1 c. sugar"", ""1 egg"", ""1...","[""Mix together for 2 or 3 minutes. Put into gr...",www.cookbooks.com/Recipe-Details.aspx?id=56276,Gathered,"[""yams"", ""sugar"", ""egg"", ""milk"", ""vanilla flav...","[(yams, 1.0, large can), (sugar, 1.0, cup), (e..."
